**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Part 5 | Session 29b: 프로젝트 학습 수행 (LoRA 파인튜닝)

## 🎯 모델 학습 실행

이 노트북에서는 수집/정제한 데이터를 사용하여 **LoRA/QLoRA 기반 파인튜닝**을 수행합니다.

### 학습 파이프라인
```
환경 설정 → 모델 로드 → 데이터 로드 → LoRA 설정 → 학습 → (DPO) → 분석 → 저장
```

### 사전 준비
- `33_project_data_pipeline.ipynb`에서 생성한 `train.json`, `eval.json`
- `project_plan.json`에 저장된 모델 설정

### RTX 4060 안전 기본값
| 파라미터 | 안전값 | 설명 |
|---------|--------|------|
| batch_size | 1~2 | OOM 방지 |
| gradient_accumulation_steps | 8 | 실효 배치=8~16 |
| max_seq_length | 1024 | VRAM 절약 |
| fp16 / bf16 | fp16 | Turing/Ada 호환 |
| gradient_checkpointing | True | VRAM 절약 |

In [ ]:
# =====================================================
# 📦 라이브러리 임포트
# =====================================================
import torch
import gc
import os
import json
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

print("✅ 라이브러리 임포트 완료")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")


In [ ]:
# =====================================================
# 📂 프로젝트 계획 + 데이터 로드
# =====================================================
OUTPUT_DIR = "./output/project"

with open(os.path.join(OUTPUT_DIR, "project_plan.json"), "r", encoding="utf-8") as f:
    project_plan = json.load(f)

with open(os.path.join(OUTPUT_DIR, "train_data.json"), "r", encoding="utf-8") as f:
    train_data = json.load(f)

MODEL_NAME = project_plan["model_config"]["base_model"]
print(f"✅ 프로젝트 계획 로드: {project_plan['project_name']}")
print(f"📌 모델: {MODEL_NAME}")
print(f"📌 학습 데이터: {len(train_data)}건")


---
## 1️⃣ 환경 설정 및 모델 로드

> 💡 **Hint**: QLoRA를 사용하면 7B 모델도 RTX 4060에서 학습 가능합니다. 4bit 양자화로 모델을 로드합니다.

In [ ]:
# =====================================================
# 🤖 모델 로드 (FP16)
# =====================================================
print(f"⏳ 모델 로딩 중: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"✅ 모델 로드 완료 (FP16)")
print(f"📌 파라미터: {model.num_parameters():,}")
print_gpu_memory("모델 로드 후")


---
## 2️⃣ 데이터 로드

이전 노트북에서 준비한 학습 데이터를 로드합니다.

In [ ]:
# =====================================================
# 📝 학습 데이터 포맷팅
# =====================================================
def format_messages(item):
    text = tokenizer.apply_chat_template(
        item["messages"], tokenize=False, add_generation_prompt=False
    )
    return text

formatted_texts = [format_messages(item) for item in train_data]
dataset = Dataset.from_dict({"text": formatted_texts})

print(f"✅ 데이터 포맷팅 완료: {len(dataset)}건")
print(f"\n--- 예시 ---")
print(dataset[0]["text"][:300])


In [ ]:
# =====================================================
# 📋 학습 전 응답 수집 (Before)
# =====================================================
EVAL_PROMPTS = project_plan["eval_prompts"]

model.eval()
before_responses = []
for prompt in EVAL_PROMPTS:
    messages = [
        {"role": "system", "content": "당신은 AI/ML 분야의 전문가입니다."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    before_responses.append(response)
    print(f"🔹 {prompt[:40]}...")
    print(f"   {response[:100]}")
    print()

print(f"✅ Before 응답 {len(before_responses)}건 수집 완료")


---
## 3️⃣ LoRA/QLoRA 설정

> 💡 **Hint**: LoRA rank(r)가 높을수록 표현력이 좋지만 메모리를 더 사용합니다. RTX 4060에서는 r=16~32가 적당합니다.

In [ ]:
# =====================================================
# 🔧 LoRA 설정 및 적용
# =====================================================
lora_config = LoraConfig(
    r=project_plan["model_config"]["lora_r"],
    lora_alpha=project_plan["model_config"]["lora_alpha"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model.train()
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print_gpu_memory("LoRA 적용 후")


---
## 4️⃣ 학습 실행 (SFTTrainer)

### Before: 학습 전 모델 출력 확인
학습 전에 모델의 현재 출력을 확인하여 Before/After 비교를 준비합니다.

In [ ]:
# =====================================================
# 🚀 SFTTrainer 설정
# =====================================================
tc = project_plan["training_config"]

sft_config = SFTConfig(
    output_dir=os.path.join(OUTPUT_DIR, "checkpoints"),
    num_train_epochs=tc["num_epochs"],
    per_device_train_batch_size=tc["batch_size"],
    gradient_accumulation_steps=tc["gradient_accumulation"],
    learning_rate=tc["learning_rate"],
    fp16=True,
    max_length=tc["max_length"],
    gradient_checkpointing=True,
    logging_steps=1,
    save_strategy="epoch",
    save_total_limit=2,
    dataset_text_field="text",
    report_to="none",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("✅ SFTTrainer 초기화 완료")
print(f"📌 에포크: {tc['num_epochs']}, 배치: {tc['batch_size']}, LR: {tc['learning_rate']}")
print_gpu_memory("학습 시작 전")


In [ ]:
# =====================================================
# 🔥 학습 실행
# =====================================================
print("🚀 학습 시작!")
print("="*60)

start_time = time.time()
train_result = trainer.train()
training_time = time.time() - start_time

print("="*60)
print("✅ 학습 완료!")
print(f"📌 Total steps: {train_result.global_step}")
print(f"📌 Training loss: {train_result.training_loss:.4f}")
print(f"📌 학습 시간: {training_time:.1f}초")
print_gpu_memory("학습 완료 후")


In [ ]:
# =====================================================
# 📊 학습 전후 비교 (Before vs After)
# =====================================================
model.eval()

print("📊 학습 전후 비교")
print("="*60)

after_responses = []
for i, prompt in enumerate(EVAL_PROMPTS):
    messages = [
        {"role": "system", "content": "당신은 AI/ML 분야의 전문가입니다."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    after = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    after_responses.append(after)
    
    changed = before_responses[i].strip() != after.strip()
    print(f"\n{'='*60}")
    print(f"🔹 {prompt}")
    print(f"  [Before] {before_responses[i][:150]}")
    print(f"  [After]  {after[:150]}")
    print(f"  {'📌 변화 감지!' if changed else '⚪ 변화 없음'}")


---
## 5️⃣ (Optional) DPO 학습

SFT 후 **DPO(Direct Preference Optimization)**를 추가로 수행하여 모델을 개선할 수 있습니다.

> ⚠️ DPO를 위해서는 **Preference 데이터** (chosen/rejected 쌍)가 필요합니다.
> 이 단계는 선택사항입니다.

In [ ]:
# =====================================================
# 💾 LoRA 어댑터 저장
# =====================================================
save_path = os.path.join(OUTPUT_DIR, "lora_adapter")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

total_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, fns in os.walk(save_path)
    for f in fns
)
print(f"✅ 어댑터 저장 완료: {save_path}")
print(f"📌 크기: {total_size / 1024 / 1024:.1f} MB")


---
## 6️⃣ 학습 로그 분석

학습 과정의 loss 변화를 분석합니다.

In [ ]:
# =====================================================
# 📈 학습 로그 분석
# =====================================================

import matplotlib.pyplot as plt

# 학습 로그 추출
log_history = trainer.state.log_history

# Train loss 추출
train_steps = [log["step"] for log in log_history if "loss" in log]
train_losses = [log["loss"] for log in log_history if "loss" in log]

# Eval loss 추출
eval_steps = [log["step"] for log in log_history if "eval_loss" in log]
eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]

# 그래프 그리기
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Train Loss
axes[0].plot(train_steps, train_losses, "b-", alpha=0.7, label="Train Loss")
axes[0].set_xlabel("Steps")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Eval Loss
if eval_losses:
    axes[1].plot(eval_steps, eval_losses, "r-o", alpha=0.7, label="Eval Loss")
    axes[1].set_xlabel("Steps")
    axes[1].set_ylabel("Loss")
    axes[1].set_title("Evaluation Loss")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "Eval 데이터 없음", ha="center", va="center")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_loss.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n📊 최종 Train Loss: {train_losses[-1]:.4f}")
if eval_losses:
    print(f"📊 최종 Eval Loss: {eval_losses[-1]:.4f}")

---
## 7️⃣ 모델 저장

학습된 모델을 저장합니다.

In [ ]:
# =====================================================
# 💾 LoRA 어댑터 저장
# =====================================================
save_path = os.path.join(OUTPUT_DIR, "lora_adapter")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

total_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, fns in os.walk(save_path)
    for f in fns
)
print(f"✅ 어댑터 저장 완료: {save_path}")
print(f"📌 크기: {total_size / 1024 / 1024:.1f} MB")


In [ ]:
# =====================================================
# 🎮 GPU 메모리 정리
# =====================================================

import gc

# 메모리 해제
del trainer
gc.collect()
torch.cuda.empty_cache()

print(f"🎮 GPU 메모리 정리 완료")
print(f"🎮 현재 사용량: {torch.cuda.memory_allocated(0)/1024**3:.2f}GB")

---
## 📝 정리

### 이번 세션에서 완료한 것
- ✅ 모델 로드 (LoRA/QLoRA/FFT)
- ✅ 학습 데이터 포매팅
- ✅ LoRA 설정 및 학습 실행
- ✅ (Optional) DPO 강화학습
- ✅ 학습 로그 분석 (Loss 그래프)
- ✅ Before/After 비교
- ✅ 모델 저장

### 산출물
- 📁 `my_project/models/final_model/` - LoRA 어댑터 (또는 전체 모델)
- 📁 `my_project/models/merged_model/` - 병합된 모델
- 📁 `my_project/models/sft_*/training_loss.png` - Loss 그래프
- 📁 `my_project/models/sft_*/before_after_comparison.json` - 비교 결과

### 하이퍼파라미터 튜닝 가이드
| 증상 | 해결 방법 |
|------|----------|
| Train loss 안 줄어듦 | learning_rate 올리기 (5e-4), 에포크 늘리기 |
| Eval loss 올라감 (오버피팅) | 에포크 줄이기, dropout 올리기, 데이터 늘리기 |
| OOM 에러 | batch_size=1, max_seq_length=512, gradient_checkpointing=True |
| 품질이 낮음 | 데이터 품질 확인, r값 올리기, 타겟 모듈 추가 |

### 다음 노트북
👉 **Session 30a** (`30a_evaluation.ipynb`): 프로젝트 성능 평가 및 반복 개선